In [3]:
!pip install torch transformers datasets rouge-score sacrebleu -q
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPTNeoForCausalLM, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn.utils.rnn import pad_sequence
from rouge_score import rouge_scorer
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
# ---------- MoE Layer Definition ----------
class MoEFeedForward(nn.Module):
    def __init__(self, hidden_size, ff_size, num_experts=4, k=2):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, ff_size),
                nn.GELU(),
                nn.Linear(ff_size, hidden_size)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(hidden_size, num_experts)

    def forward(self, x):
        batch, seq_len, hidden_size = x.size()
        gate_scores = self.gate(x)
        topk_vals, topk_idx = torch.topk(gate_scores, self.k, dim=-1)
        topk_softmax = F.softmax(topk_vals, dim=-1)
        output = torch.zeros_like(x)

        for i in range(self.k):
            idx = topk_idx[..., i]
            weight = topk_softmax[..., i]
            expert_out = torch.zeros_like(x)
            for expert_id, expert in enumerate(self.experts):
                mask = (idx == expert_id).float().unsqueeze(-1)
                expert_out += expert(x) * mask
            output += expert_out * weight.unsqueeze(-1)
        return output

In [5]:
# ---------- Convert GPT-Neo blocks to MoE ----------
def convert_gptneo_to_moe(model, num_experts=4, k=2):
    for block in model.transformer.h:
        hidden_size = model.config.hidden_size
        ff_size = block.mlp.c_fc.out_features
        block.mlp = MoEFeedForward(hidden_size, ff_size, num_experts=num_experts, k=k)
    return model

In [6]:
# ---------- Load tokenizer and dense GPT-Neo ----------
tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
tokenizer.pad_token = tokenizer.eos_token
model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M").to(device)
model.eval()

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [7]:
# ---------- Load and format datasets ----------
sst2 = load_dataset("glue", "sst2", split="validation")
cnn_dm = load_dataset("cnn_dailymail", "3.0.0", split="validation")
qqp = load_dataset("glue", "qqp", split="validation")
squad = load_dataset("squad", split="validation")

def format_sst2(example):
    return {"prompt": f"Task: sentiment. Text: {example['sentence']} Answer:", "label_text": str(example['label'])}

def format_cnn_dm(example):
    return {"prompt": f"Task: summarize. Text: {example['article']} Summary:", "label_text": example['highlights']}

def format_qqp(example):
    return {"prompt": f"Task: paraphrase. Text1: {example['question1']} Text2:", "label_text": example['question2'] or ""}

def format_squad(example):
    answer = example['answers']['text'][0] if example['answers']['text'] else ""
    return {"prompt": f"Task: question answering. Context: {example['context']} Question: {example['question']} Answer:", "label_text": answer}

sst2_prompts = sst2.map(format_sst2, remove_columns=sst2.column_names)
cnn_dm_prompts = cnn_dm.map(format_cnn_dm, remove_columns=cnn_dm.column_names)
qqp_prompts = qqp.map(format_qqp, remove_columns=qqp.column_names)
squad_prompts = squad.map(format_squad, remove_columns=squad.column_names)

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [8]:
# ---------- Text generation & evaluation ----------
def generate_text(prompt, max_new_tokens=50, max_input_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens).to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def classify_sentiment(pred_text):
    pred_text = pred_text.lower()
    if any(word in pred_text for word in ["positive", "great", "good", "excellent", "love"]):
        return "1"
    elif any(word in pred_text for word in ["negative", "bad", "terrible", "hate", "poor"]):
        return "0"
    else:
        return "1"

def evaluate(dataset, task_type="classification"):
    results = []
    for i in range(10):
        ex = dataset[i]
        pred_text = generate_text(ex["prompt"])
        label = ex.get("label_text", "")
        if task_type == "classification":
            pred_label = classify_sentiment(pred_text)
            results.append({"prompt": ex["prompt"], "label": label, "prediction": pred_label})
        else:
            results.append({"prompt": ex["prompt"], "label": label, "prediction": pred_text})
    if task_type == "classification":
        correct = sum([r["prediction"] == r["label"] for r in results])
        return correct / len(results), results
    else:
        rouge = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
        rouge_scores = [rouge.score(r["label"], r["prediction"]) for r in results]
        bleu_scores = [sacrebleu.corpus_bleu([r["prediction"]], [[r["label"]]]).score for r in results]
        return rouge_scores, bleu_scores, results


In [9]:
# ---------- Encode for fine-tuning ----------
MAX_EXAMPLES = 50
datasets_small = [
    sst2_prompts.select(range(MAX_EXAMPLES)),
    cnn_dm_prompts.select(range(MAX_EXAMPLES)),
    qqp_prompts.select(range(MAX_EXAMPLES)),
    squad_prompts.select(range(MAX_EXAMPLES))
]

all_data = []
for dataset in datasets_small:
    all_data.extend(dataset)

def encode_example(ex, max_input=512, max_output=128):
    prompt_ids = tokenizer(ex["prompt"], truncation=True, max_length=max_input, return_tensors="pt").input_ids[0]
    label_ids = tokenizer(ex["label_text"], truncation=True, max_length=max_output, return_tensors="pt").input_ids[0]
    input_ids = torch.cat([prompt_ids, label_ids], dim=0)
    labels = torch.cat([torch.full_like(prompt_ids, -100), label_ids], dim=0)
    return {"input_ids": input_ids, "labels": labels}

encoded_data = [encode_example(ex) for ex in all_data]

def collate_fn(batch):
    input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    labels[labels == tokenizer.pad_token_id] = -100
    return {"input_ids": input_ids.to(device), "labels": labels.to(device)}

loader = DataLoader(encoded_data, batch_size=2, shuffle=True, collate_fn=collate_fn)
optimizer = AdamW(model.parameters(), lr=5e-5)


In [10]:
# ---------- Step 1: Evaluate dense GPT-Neo ----------
sent_acc_dense, _ = evaluate(sst2_prompts, "classification")
summ_rouge_dense, summ_bleu_dense, _ = evaluate(cnn_dm_prompts, "generation")
para_rouge_dense, para_bleu_dense, _ = evaluate(qqp_prompts, "generation")
qa_rouge_dense, qa_bleu_dense, _ = evaluate(squad_prompts, "generation")

print("Dense GPT-Neo before MoE conversion:")
print("Sentiment Accuracy:", sent_acc_dense)
print("Summarization example ROUGE:", summ_rouge_dense[0], "BLEU:", summ_bleu_dense[0])
print("Paraphrase example ROUGE:", para_rouge_dense[0], "BLEU:", para_bleu_dense[0])
print("QA example ROUGE:", qa_rouge_dense[0], "BLEU:", qa_bleu_dense[0])

Dense GPT-Neo before MoE conversion:
Sentiment Accuracy: 0.5
Summarization example ROUGE: {'rouge1': Score(precision=0.03862660944206009, recall=0.782608695652174, fmeasure=0.0736196319018405), 'rouge2': Score(precision=0.017204301075268817, recall=0.36363636363636365, fmeasure=0.03285420944558522), 'rougeL': Score(precision=0.032188841201716736, recall=0.6521739130434783, fmeasure=0.061349693251533735)} BLEU: 0.7393765743390373
Paraphrase example ROUGE: {'rouge1': Score(precision=0.08888888888888889, recall=0.8, fmeasure=0.15999999999999998), 'rouge2': Score(precision=0.045454545454545456, recall=0.5, fmeasure=0.08333333333333334), 'rougeL': Score(precision=0.08888888888888889, recall=0.8, fmeasure=0.15999999999999998)} BLEU: 2.878588551893216
QA example ROUGE: {'rouge1': Score(precision=0.011428571428571429, recall=1.0, fmeasure=0.02259887005649718), 'rouge2': Score(precision=0.005747126436781609, recall=1.0, fmeasure=0.011428571428571429), 'rougeL': Score(precision=0.011428571428571

In [11]:
# ---------- Step 2: Convert to top-2 MoE ----------
model = convert_gptneo_to_moe(model, num_experts=4, k=2).to(device)

In [12]:
# ---------- Step 3: Fine-tune MoE GPT-Neo ----------
model.train()
for batch in loader:
    optimizer.zero_grad()
    outputs = model(input_ids=batch["input_ids"], labels=batch["labels"])
    loss = outputs.loss
    loss.backward()
    optimizer.step()

model.eval()

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MoEFeedForward(
          (experts): ModuleList(
            (0-3): 4 x Sequential(
              (0): Linear(in_fe

In [14]:
print(model.transformer.h[0].mlp)


MoEFeedForward(
  (experts): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=768, out_features=3072, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=3072, out_features=768, bias=True)
    )
  )
  (gate): Linear(in_features=768, out_features=4, bias=True)
)


In [13]:
# ---------- Step 4: Evaluate MoE GPT-Neo ----------
sent_acc_moe, _ = evaluate(sst2_prompts, "classification")
summ_rouge_moe, summ_bleu_moe, _ = evaluate(cnn_dm_prompts, "generation")
para_rouge_moe, para_bleu_moe, _ = evaluate(qqp_prompts, "generation")
qa_rouge_moe, qa_bleu_moe, _ = evaluate(squad_prompts, "generation")

print("MoE GPT-Neo after fine-tuning:")
print("Sentiment Accuracy:", sent_acc_moe)
print("Summarization example ROUGE:", summ_rouge_moe[0], "BLEU:", summ_bleu_moe[0])
print("Paraphrase example ROUGE:", para_rouge_moe[0], "BLEU:", para_bleu_moe[0])
print("QA example ROUGE:", qa_rouge_moe[0], "BLEU:", qa_bleu_moe[0])

MoE GPT-Neo after fine-tuning:
Sentiment Accuracy: 0.5
Summarization example ROUGE: {'rouge1': Score(precision=0.04081632653061224, recall=0.782608695652174, fmeasure=0.07758620689655171), 'rouge2': Score(precision=0.01818181818181818, recall=0.36363636363636365, fmeasure=0.03463203463203463), 'rougeL': Score(precision=0.034013605442176874, recall=0.6521739130434783, fmeasure=0.06465517241379311)} BLEU: 0.7393765743390373
Paraphrase example ROUGE: {'rouge1': Score(precision=0.06666666666666667, recall=0.8, fmeasure=0.12307692307692308), 'rouge2': Score(precision=0.03389830508474576, recall=0.5, fmeasure=0.06349206349206349), 'rougeL': Score(precision=0.06666666666666667, recall=0.8, fmeasure=0.12307692307692308)} BLEU: 2.6482245289541764
QA example ROUGE: {'rouge1': Score(precision=0.0136986301369863, recall=1.0, fmeasure=0.027027027027027025), 'rouge2': Score(precision=0.006896551724137931, recall=1.0, fmeasure=0.0136986301369863), 'rougeL': Score(precision=0.0136986301369863, recall=